# HW3 — Two retrievers and a model: what RAG actually sends

**Name:** Aruzhan Abuova
**Student ID:** S23068103
**Group:** CSS4007-ENG-8
**Repository:** https://github.com/aruzhanabuova-png/Hw3-aru_aru.git

## AI tool disclosure

State which AI tools you used and for what. Expected and fine; undisclosed use is not.

> I used ChatGPT to understand the assignment requirements, explain BM25, cosine similarity, embeddings and RAG, and help with Python code and debugging. I reviewed and tested the code and results myself.

Run this notebook **top to bottom** before you push. A cell with no saved output earns nothing.
The instructions for every section are in `README.md`.

---
# Part A — relevance by hand (1 pt)

| | Text |
|---|---|
| D1 | Large language models can generate text and answer questions. |
| D2 | Retrieval-Augmented Generation retrieves relevant documents before generating an answer. |
| D3 | Embeddings represent text as numerical vectors. |
| Query | How does RAG retrieve information? |

`k1 = 1.5`, `b = 0.75`, `idf(t) = ln((N − n(t) + 0.5) / (n(t) + 0.5))`. Tokenise as README §3 says.

## A1 (a) — BM25 as written

**Tokens of each document, and its length:**

| Doc | Tokens | \|d\| |
|---|---|---|
| D1 | large, language, models, can, generate, text, and, answer, questions | 9 |
| D2 | retrieval, augmented, generation, retrieves, relevant, documents, before, generating, an, answer | 10 |
| D3 | embeddings, represent, text, as, numerical, vectors | 6 |

**avgdl =** (9 + 10 + 6) / 3 = 8.33

**Query terms:** how, does, rag, retrieve, information

**TF, DF and IDF of each query term:**

| Query term | f(t, D1) | f(t, D2) | f(t, D3) | n(t) (DF) | idf(t) |
|---|---:|---:|---:|---:|---:|
| how | 0 | 0 | 0 | 0 | 1.946 |
| does | 0 | 0 | 0 | 0 | 1.946 |
| rag | 0 | 0 | 0 | 0 | 1.946 |
| retrieve | 0 | 0 | 0 | 0 | 1.946 |
| information | 0 | 0 | 0 | 0 | 1.946 |

**IDF calculation:**

N = 3, DF = 0

IDF = ln((3 - 0 + 0.5) / (0 + 0.5))

IDF = ln(7) = 1.946

**BM25 of each document** (write out the sum for every document, one line per term that contributes):

 For all five query terms, the term frequency is 0 in every document.

k1 = 1.5, b = 0.75, avgdl = 8.33

D1: BM25 = 5 × (1.946 × 0 × 2.5 / (0 + 1.5 × (1 - 0.75 + 0.75 × 9/8.33))) = 0

D2: BM25 = 5 × (1.946 × 0 × 2.5 / (0 + 1.5 × (1 - 0.75 + 0.75 × 10/8.33))) = 0

D3: BM25 = 5 × (1.946 × 0 × 2.5 / (0 + 1.5 × (1 - 0.75 + 0.75 × 6/8.33))) = 0

| Document | BM25 Score | Rank |
|---|---:|---|
| D1 | 0 | 1 (tie) |
| D2 | 0 | 1 (tie) |
| D3 | 0 | 1 (tie) |

**Explanation:**
All three documents have a BM25 score of 0 because none of the exact query terms appear in the documents. Therefore, BM25 cannot identify a single best document without stemming.

## A1 (b) — BM25 with the stems from README §3

**Stemmed tokens of each document, and its length:**

| Doc | Tokens | \|d\| |
|---|---|---:|
| D1 | large, language, models, can, gener, text, and, answer, questions | 9 |
| D2 | retriev, augmented, gener, retriev, relevant, documents, before, gener, an, answer | 10 |
| D3 | embeddings, represent, text, as, numerical, vectors | 6 |

**avgdl =** (9 + 10 + 6) / 3 = 8.33

**Stemmed query terms:** how, doe, rag, retriev, inform
 
| Query term | f(t, D1) | f(t, D2) | f(t, D3) | n(t) (DF) | idf(t) |
|---|---:|---:|---:|---:|---:|
| how | 0 | 0 | 0 | 0 | 1.946 |
| doe | 0 | 0 | 0 | 0 | 1.946 |
| rag | 0 | 0 | 0 | 0 | 1.946 |
| retriev | 0 | 2 | 0 | 1 | 0.511 |
| inform | 0 | 0 | 0 | 0 | 1.946 |

**IDF calculation:**

N = 3

For how, doe, rag, inform:

DF = 0

IDF = ln((3 - 0 + 0.5) / (0 + 0.5)) = ln(7) ≈ 1.946

For retriev:

DF = 1

IDF = ln((3 - 1 + 0.5) / (1 + 0.5))

IDF = ln(2.5 / 1.5) ≈ 0.511 

**BM25 of each document**, worked:

**BM25 calculation:**

k1 = 1.5, b = 0.75, avgdl = 8.33

Only the term "retriev" contributes to the BM25 score because all other query terms have TF = 0.

D1: BM25 = 0

D2: BM25 = (0.511 × 2 × 2.5) / (2 + 1.5 × (1 - 0.75 + 0.75 × 10/8.33)) ≈ 0.6858

D3: BM25 = 0

| Document | BM25 Score | Rank |
|---|---:|---:|
| D1 | 0 | 2 (tie) |
| D2 | 0.6858 | 1 |
| D3 | 0 | 2 (tie) |

**Which document does BM25 retrieve first, in (a) and in (b)? Why?**

In (a), all three documents have a BM25 score of 0, so there is no clear winner. The query term "retrieve" does not exactly match "retrieval" or "retrieves" in D2.

In (b), D2 ranks first with a BM25 score of approximately 0.6858. After stemming, "retrieve", "retrieval", and "retrieves" become "retriev". This allows BM25 to find two matching terms in D2.

## A2 — cosine similarity

```
Query = [0.8, 0.2, 0.5]
D1 = [0.7, 0.1, 0.4]   D2 = [0.9, 0.3, 0.6]   D3 = [0.1, 0.8, 0.2]
```

| Pair | Dot product (worked) | ‖Query‖ | ‖D‖ | Cosine |
|---|---|---:|---:|---:|
| Query · D1 | 0.8×0.7 + 0.2×0.1 + 0.5×0.4 = 0.78 | 0.9644 | 0.8124 | 0.9956 |
| Query · D2 | 0.8×0.9 + 0.2×0.3 + 0.5×0.6 = 1.08 | 0.9644 | 1.1225 | 0.9977 |
| Query · D3 | 0.8×0.1 + 0.2×0.8 + 0.5×0.2 = 0.34 | 0.9644 | 0.8307 | 0.4244 |

**Vector magnitudes:**

‖Query‖ = √(0.8² + 0.2² + 0.5²) = √0.93 ≈ 0.9644

‖D1‖ = √(0.7² + 0.1² + 0.4²) = √0.66 ≈ 0.8124

‖D2‖ = √(0.9² + 0.3² + 0.6²) = √1.26 ≈ 1.1225

‖D3‖ = √(0.1² + 0.8² + 0.2²) = √0.69 ≈ 0.8307

**Cosine similarity calculation:**

D1: Cosine = 0.78 / (√0.93 × √0.66) ≈ 0.9956

D2: Cosine = 1.08 / (√0.93 × √1.26) ≈ 0.9977

D3: Cosine = 0.34 / (√0.93 × √0.69) ≈ 0.4244

| Document | Cosine Similarity | Rank |
|---|---:|---:|
| D1 | 0.9956 | 2 |
| D2 | 0.9977 | 1 |
| D3 | 0.4244 | 3 |

**Which document is most similar to the query?**

> D2 is the most similar document to the query, with a cosine similarity of approximately 0.9977.

**Is the ranking the same as BM25's?** Compare it with both A1 rankings.

> The cosine similarity ranking is not exactly the same as BM25. In A1 (a), all documents have a BM25 score of 0, so they are tied. In A1 (b), D2 ranks first, while D1 and D3 are tied. With cosine similarity, D2 ranks first, followed by D1 and D3. Therefore, cosine similarity and BM25 with stemming agree on the best document, but their full rankings are different.

**Why can BM25 and embedding retrieval disagree?** Point at the words of this query and these documents.

> BM25 and embedding retrieval can disagree because BM25 depends on matching words, while embeddings represent semantic similarity. For example, the query contains "retrieve", but D2 contains "retrieval" and "retrieves". Without stemming, BM25 cannot match these words. Embedding retrieval can recognize semantic relationships even when the exact words are different. In this example, cosine similarity ranks D2 first because its vector is the most similar to the query vector.

Optional: check your Part A arithmetic here. The working above must still be written out by hand.

---
# Part B — the same two retrievers, from libraries (1.5 pt)

Load `data/corpus.jsonl` and `data/queries.json`.

In [52]:
import json

# Открываем файл с документами и загружаем его.
with open("data/corpus.jsonl", "r", encoding="utf-8") as f:
    corpus = [json.loads(line) for line in f]

# Load 8 fixed queries
with open("data/queries.json", "r", encoding="utf-8") as f:
    fixed_queries = json.load(f)["queries"]

print("Number of documents:", len(corpus))
print("Number of queries:", len(fixed_queries))

print("\nFirst document:")
print(corpus[0])

# Показываем первый вопрос и список документов, которые считаются правильными для этого вопроса.
print("\nFirst query:")
print(fixed_queries[0])

Number of documents: 74
Number of queries: 8

First document:
{'id': 'KZ-01', 'source': 'wikipedia', 'title': 'Kazakhstan (part 1)', 'lang': 'en', 'url': 'https://en.wikipedia.org/wiki/Kazakhstan', 'text': "Kazakhstan, officially the Republic of Kazakhstan, is a landlocked country situated primarily in Central Asia, with a portion of its territory extending into Eastern Europe. It borders Russia to the north and west, China to the east, Kyrgyzstan to the southeast, Uzbekistan to the south, and Turkmenistan to the southwest, and it has a coastline along the Caspian Sea. The capital is Astana, and the country's largest city and principal cultural and economic center is Almaty, which served as the capital until 1997."}

First query:
{'id': 'Q-01', 'text': 'What is form GR-305 used for?', 'relevant': ['GO-08', 'GO-12', 'GO-23']}


## B1 — BM25 with `rank_bm25`

**My tokenisation:** (lowercase? regex? stemming? stop words?)

> I use lowercase text and regex tokenization (`\b\w+\b`). I do not use stemming or remove stop words. BM25 matches words based on their exact forms.

In [53]:
# Подготавливаем документы для BM25: разбиваем текст на слова,
# создаём поисковый индекс и находим три наиболее подходящих документа для каждого вопроса.
import re
from rank_bm25 import BM25Okapi

# Разбиваем текст на отдельные слова и переводим их в нижний регистр.
def tokenize(text):
    return re.findall(r"\b\w+\b", text.lower())

# Разбиваем тексты всех 74 документов на слова для поиска.
tokenized_corpus = [tokenize(doc["text"]) for doc in corpus]

# Создаём BM25-индекс для поиска документов по словам.
bm25 = BM25Okapi(tokenized_corpus, k1=1.5, b=0.75)

print("BM25 index created successfully!")
print("Number of indexed documents:", len(tokenized_corpus))
print("First document tokens:", tokenized_corpus[0][:15])


# Ищем три наиболее подходящих документа для каждого вопроса
# и проверяем, попал ли хотя бы один правильный документ в топ-3.

bm25_results = {}

for query in fixed_queries:
    query_id = query["id"]
    query_text = query["text"]
    relevant_ids = query["relevant"]

    # Рассчитываем оценки релевантности документов для текущего вопроса.
    scores = bm25.get_scores(tokenize(query_text))

    # Выбираем индексы трёх документов с самыми высокими оценками.
    top_indices = sorted(
        range(len(scores)),
        key=lambda i: scores[i],
        reverse=True
    )[:3]

    # Сохраняем ID и оценки трёх найденных документов.
    bm25_results[query_id] = [
        {
            "id": corpus[i]["id"],
            "score": float(scores[i])
        }
        for i in top_indices
    ]

    # Проверяем Hit@3: найден ли хотя бы один правильный документ в топ-3.
    hit = any(corpus[i]["id"] in relevant_ids for i in top_indices)

    print(f"\n{query_id}: {query_text}")
    
    # Выводим номер документа, его ID, оценку и начало текста.
    for rank, i in enumerate(top_indices, start=1):
        first_line = corpus[i]["text"].splitlines()[0][:120]
        print(
            f"{rank}. {corpus[i]['id']} | "
            f"Score: {scores[i]:.4f} | "
            f"{first_line}"
        )

    print(f"Hit@3: {hit}")

BM25 index created successfully!
Number of indexed documents: 74
First document tokens: ['kazakhstan', 'officially', 'the', 'republic', 'of', 'kazakhstan', 'is', 'a', 'landlocked', 'country', 'situated', 'primarily', 'in', 'central', 'asia']

Q-01: What is form GR-305 used for?
1. GO-08 | Score: 15.8255 | Form GR-305 is used to appeal a decision of the Grant Office. It is the appeal form and nothing else: it cannot be used 
2. GO-12 | Score: 8.1953 | A student who disagrees with a decision may appeal within 14 calendar days of the date on the decision letter, using for
3. GO-06 | Score: 8.0700 | Form GR-104 is the application form for the need-based study grant. It is the only way to apply; an email or a letter is
Hit@3: True

Q-02: My family is poor. Is there financial help for students like me?
1. KZ-17 | Score: 8.7262 | The lake currently covers about 16,400 km2 (6,300 sq mi). However, like the Aral Sea, it is shrinking due to diversion a
2. GO-20 | Score: 8.3105 | The need-based gr

In [54]:
# Inspect documents for our two custom queries

for doc in corpus:
    if doc["id"] in ["GO-08", "GO-14"]:
        print("ID:", doc["id"])
        print("Title:", doc["title"])
        print("Text:", doc["text"])
        print("-" * 60)

ID: GO-08
Title: Form GR-305: appeals
Text: Form GR-305 is used to appeal a decision of the Grant Office. It is the appeal form and nothing else: it cannot be used to apply for the first time or to report a change in circumstances. On GR-305 the student states which part of the decision they believe is wrong and which rule in this handbook supports their case.
------------------------------------------------------------
ID: GO-14
Title: Өтініш беру мерзімі және тәртібі
Text: 2026 жылғы грантқа өтініштер 1 қыркүйектен бастап 15 қазанға дейін студенттік портал арқылы қабылданады. Өтініш беру үшін GR-104 нысанын толтырып, транскрипт пен жеке куәліктің көшірмесін тіркеу керек. Мерзімі өткен өтініштер қаралмайды. Сұрақтарыңыз болса, Грант бөліміне хабарласыңыз.
------------------------------------------------------------


### My own queries (before retrieval)

**Q-09:** What is the purpose of form GR-305?

- Relevant document: GO-08
- Prediction: BM25 may perform better because the query contains the exact form number GR-305.

**Q-10:** Грантқа құжаттарды қай күнге дейін өткізуге болады?

- Relevant document: GO-14
- Prediction: EmbeddingGemma may perform better because the query uses different words to express the same meaning as the document.

In [55]:
# Создаём два собственных вопроса с ID правильных документов.
# Объединяем их с 8 готовыми вопросами и проверяем общее количество.
my_queries = [
    {
        "id": "Q-09",
        "text": "What is the purpose of form GR-305?",
        "relevant": ["GO-08"]
    },
    {
        "id": "Q-10",
        "text": "Грантқа құжаттарды қай күнге дейін өткізуге болады?",
        "relevant": ["GO-14"]
    }
]

all_queries = fixed_queries + my_queries

print("Total queries:", len(all_queries))

for query in my_queries:
    print(query)


# Используем BM25 для поиска трёх наиболее подходящих документов по нашим вопросам.
# Сохраняем результаты и проверяем, попал ли правильный документ в топ-3.

for query in my_queries:
    scores = bm25.get_scores(tokenize(query["text"]))

    top_indices = sorted(
        range(len(scores)),
        key=lambda i: scores[i],
        reverse=True
    )[:3]

    bm25_results[query["id"]] = [
        {"id": corpus[i]["id"], "score": float(scores[i])}
        for i in top_indices
    ]

    print(f"\n{query['id']}: {query['text']}")

    for rank, i in enumerate(top_indices, start=1):
        print(
            f"{rank}. {corpus[i]['id']} | "
            f"Score: {scores[i]:.4f} | "
            f"{corpus[i]['text'].splitlines()[0][:120]}"
        )

    hit = any(corpus[i]["id"] in query["relevant"] for i in top_indices)
    print(f"Hit@3: {hit}")

Total queries: 10
{'id': 'Q-09', 'text': 'What is the purpose of form GR-305?', 'relevant': ['GO-08']}
{'id': 'Q-10', 'text': 'Грантқа құжаттарды қай күнге дейін өткізуге болады?', 'relevant': ['GO-14']}

Q-09: What is the purpose of form GR-305?
1. GO-08 | Score: 15.5906 | Form GR-305 is used to appeal a decision of the Grant Office. It is the appeal form and nothing else: it cannot be used 
2. GO-12 | Score: 11.4204 | A student who disagrees with a decision may appeal within 14 calendar days of the date on the decision letter, using for
3. GO-07 | Score: 9.8670 | Form GR-210 is the household income declaration. It lists every adult living in the household and the monthly income of 
Hit@3: True

Q-10: Грантқа құжаттарды қай күнге дейін өткізуге болады?
1. GO-14 | Score: 10.6979 | 2026 жылғы грантқа өтініштер 1 қыркүйектен бастап 15 қазанға дейін студенттік портал арқылы қабылданады. Өтініш беру үші
2. KZ-01 | Score: 0.0000 | Kazakhstan, officially the Republic of Kazakhstan, is a land

## B2 — EmbeddingGemma (`google/embeddinggemma-300m`) with `encode_document` / `encode_query`

In [56]:
# Подключаем Hugging Face с помощью токена для доступа к модели.
# Загружаем EmbeddingGemma, которая преобразует текст в числовые векторы для поиска по смыслу.
from huggingface_hub import login
from sentence_transformers import SentenceTransformer
from getpass import getpass

hf_token = getpass("Enter your Hugging Face token: ")
login(token=hf_token, add_to_git_credential=False)
del hf_token

model = SentenceTransformer("google/embeddinggemma-300m")

print("EmbeddingGemma model loaded successfully!")

Enter your Hugging Face token:  ········


Loading weights: 100%|██████████| 314/314 [00:00<00:00, 519.91it/s]


EmbeddingGemma model loaded successfully!


In [57]:
# Преобразуем тексты всех 74 документов в числовые векторы (embeddings).
# Эти векторы помогут находить документы по смыслу, а не только по совпадению слов.

document_texts = [doc["text"] for doc in corpus]

document_embeddings = model.encode_document(
    document_texts,
    show_progress_bar=True
)

print("Document embeddings created successfully!")
print("Shape:", document_embeddings.shape)

Batches: 100%|██████████| 3/3 [00:23<00:00,  7.85s/it]

Document embeddings created successfully!
Shape: (74, 768)


In [58]:
# Преобразуем все 10 вопросов в числовые векторы (embeddings).
# Эти векторы будем сравнивать с векторами документов, чтобы находить ответы по смыслу.

query_texts = [query["text"] for query in all_queries]

query_embeddings = model.encode_query(
    query_texts,
    show_progress_bar=True
)

print("Query embeddings created successfully!")
print("Shape:", query_embeddings.shape)

Batches: 100%|██████████| 1/1 [00:00<00:00,  1.54it/s]

Query embeddings created successfully!
Shape: (10, 768)


In [59]:
# Сравниваем векторы всех 10 вопросов с векторами 74 документов с помощью cosine similarity.
# Находим три наиболее похожих документа для каждого вопроса и сохраняем результаты.
# Проверяем Hit@3: попал ли хотя бы один правильный документ в топ-3.
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# Вычисляем сходство между вопросами и документами.
similarities = cosine_similarity(query_embeddings, document_embeddings)

embedding_results = {}

for q_idx, query in enumerate(all_queries):
    scores = similarities[q_idx]

    # Выбираем три документа с наибольшим сходством.
    top_indices = np.argsort(scores)[::-1][:3]

    embedding_results[query["id"]] = [
        {"id": corpus[i]["id"], "score": float(scores[i])}
        for i in top_indices
    ]

    print(f"\n{query['id']}: {query['text']}")

    for rank, i in enumerate(top_indices, start=1):
        first_line = corpus[i]["text"].splitlines()[0][:120]
        print(
            f"{rank}. {corpus[i]['id']} | "
            f"Score: {scores[i]:.4f} | "
            f"{first_line}"
        )

    hit = any(corpus[i]["id"] in query["relevant"] for i in top_indices)
    print(f"Hit@3: {hit}")


Q-01: What is form GR-305 used for?
1. GO-08 | Score: 0.6955 | Form GR-305 is used to appeal a decision of the Grant Office. It is the appeal form and nothing else: it cannot be used 
2. GO-12 | Score: 0.4609 | A student who disagrees with a decision may appeal within 14 calendar days of the date on the decision letter, using for
3. GO-23 | Score: 0.4505 | Если студент не согласен с решением, он может подать апелляцию по форме GR-305 в течение 14 календарных дней со дня реше
Hit@3: True

Q-02: My family is poor. Is there financial help for students like me?
1. GO-01 | Score: 0.4446 | The need-based study grant 2026 is run by the Grant Office for full-time undergraduate students. It exists so that a stu
2. GO-10 | Score: 0.4117 | Proof of income must accompany form GR-210. Accepted: a salary certificate from the employer issued within the last thre
3. GO-20 | Score: 0.4090 | The need-based grant is for full-time undergraduate students only. Part-time students, distance-learning student

## My own queries

Write them, and their `relevant` chunk ids, **before** you run them. Say which one you expect BM25 to win and which you expect embeddings to win, and why.

| Id | Query | Relevant | Expected winner, and why |
|---|---|---|---|
| Q-09 | What is the purpose of form GR-305? | GO-08 | BM25, because the query contains the exact form number GR-305. |
| Q-10 | Грантқа құжаттарды қай күнге дейін өткізуге болады? | GO-14 | EmbeddingGemma, because the query uses different words with a similar meaning. |

In [17]:
%pip install pandas

   ---------------------------------------- 0.0/9.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.6 MB ? eta -:--:--
   - -------------------------------------- 0.3/9.6 MB ? eta -:--:--
   - -------------------------------------- 0.3/9.6 MB ? eta -:--:--
   -- ------------------------------------- 0.5/9.6 MB 853.9 kB/s eta 0:00:11
   --- ------------------------------------ 0.8/9.6 MB 845.8 kB/s eta 0:00:11
   --- ------------------------------------ 0.8/9.6 MB 845.8 kB/s eta 0:00:11
   --- ------------------------------------ 0.8/9.6 MB 845.8 kB/s eta 0:00:11
   --- ------------------------------------ 0.8/9.6 MB 845.8 kB/s eta 0:00:11
   --- ------------------------------------ 0.8/9.6 MB 845.8 kB/s eta 0:00:11
   --- ------------------------------------ 0.8/9.6 MB 845.8 kB/s eta 0:00:11
   --- ------------------------------------ 0.8/9.6 MB 845.8 kB/s eta 0:00:11
   ---- -----------------------

## Results for all ten or more queries: top 3 from each retriever

In [60]:
# Сравниваем результаты поиска BM25 и EmbeddingGemma для всех 10 вопросов.
# Проверяем, какие документы попали в топ-3 и найден ли среди них правильный ответ.
# Создаём таблицу сравнения и вычисляем общий процент Hit@3 для каждого метода.
import pandas as pd

comparison = []

for query in all_queries:
    qid = query["id"]
    relevant = set(query["relevant"])

    bm25_top3 = [item["id"] for item in bm25_results[qid]]
    emb_top3 = [item["id"] for item in embedding_results[qid]]
    
    # Проверяем, есть ли правильный документ среди первых трёх.
    bm25_hit = any(doc_id in relevant for doc_id in bm25_top3)
    emb_hit = any(doc_id in relevant for doc_id in emb_top3)

    comparison.append({
        "Query": qid,
        "BM25 Top-3": ", ".join(bm25_top3),
        "BM25 Hit@3": bm25_hit,
        "EmbeddingGemma Top-3": ", ".join(emb_top3),
        "EmbeddingGemma Hit@3": emb_hit
    })

# Преобразуем результаты в таблицу.
comparison_df = pd.DataFrame(comparison)

display(comparison_df)

print("\nBM25 Hit@3:", f"{comparison_df['BM25 Hit@3'].mean():.0%}")
print("EmbeddingGemma Hit@3:", f"{comparison_df['EmbeddingGemma Hit@3'].mean():.0%}")

,Query,BM25 Top-3,BM25 Hit@3,EmbeddingGemma Top-3,EmbeddingGemma Hit@3
0,Q-01,"GO-08, GO-12, GO-06",True,"GO-08, GO-12, GO-23",True
1,Q-02,"KZ-17, GO-20, KZ-25",False,"GO-01, GO-10, GO-20",True
2,Q-03,"GO-14, KZ-01, KZ-02",True,"GO-14, GO-09, GO-11",True
3,Q-04,"GO-05, GO-17, GO-19",False,"GO-02, GO-05, GO-17",True
4,Q-05,"KZ-05, KZ-06, KZ-11",True,"KZ-05, KZ-06, KZ-03",True
5,Q-06,"KZ-37, KZ-38, KZ-03",True,"KZ-37, KZ-38, KZ-10",True
6,Q-07,"GO-21, GO-16, GO-01",False,"GO-15, GO-18, GO-01",True
7,Q-08,"GO-17, GO-05, GO-04",True,"GO-17, GO-18, GO-15",True
8,Q-09,"GO-08, GO-12, GO-07",True,"GO-08, GO-12, GO-23",True
9,Q-10,"GO-14, KZ-01, KZ-02",True,"GO-14, GO-09, GO-15",True



BM25 Hit@3: 70%
EmbeddingGemma Hit@3: 100%


## B3 — comparison

| Query | BM25 top 3 (ids) | BM25 hit@3 | Embedding top 3 (ids) | Embedding hit@3 |
|---|---|---|---|---|
| Q-01 | GO-08, GO-12, GO-06 | Yes | GO-08, GO-12, GO-23 | Yes |
| Q-02 | KZ-17, GO-20, KZ-25 | No | GO-01, GO-10, GO-20 | Yes |
| Q-03 | GO-14, KZ-01, KZ-02 | Yes | GO-14, GO-09, GO-11 | Yes |
| Q-04 | GO-05, GO-17, GO-19 | No | GO-02, GO-05, GO-17 | Yes |
| Q-05 | KZ-05, KZ-06, KZ-11 | Yes | KZ-05, KZ-06, KZ-03 | Yes |
| Q-06 | KZ-37, KZ-38, KZ-03 | Yes | KZ-37, KZ-38, KZ-10 | Yes |
| Q-07 | GO-21, GO-16, GO-01 | No | GO-15, GO-18, GO-01 | Yes |
| Q-08 | GO-17, GO-05, GO-04 | Yes | GO-17, GO-18, GO-15 | Yes |
| Q-09 | GO-08, GO-12, GO-07 | Yes | GO-08, GO-12, GO-23 | Yes |
| Q-10 | GO-14, KZ-01, KZ-02 | Yes | GO-14, GO-09, GO-15 | Yes |
| **Total** | | **7/10 (70%)** | | **10/10 (100%)** |


**Both retrievers agree:** which query, and why was it easy?

> Both retrievers found GO-08 as the top result for Q-01. The question contains the exact form number GR-305 and the document clearly explains its purpose. This makes the query easy for both methods.

**BM25 did better:** which query, and what exact string did it match that the embedding model blurred?

> BM25 did not achieve a better Hit@3 result than EmbeddingGemma on any query. However, BM25 worked well on Q-09 because it matched the exact string "GR-305". Both methods ranked GO-08 first, so this was a tie rather than a BM25 win.

**Embeddings did better:** which query, and what could BM25 not match?

> EmbeddingGemma performed better on Q-02. The question asks about financial help for a poor family, while the relevant documents use terms such as "need-based grant". BM25 could not match these different expressions well, but EmbeddingGemma understood their similar meaning and found the relevant documents.

---
# Part C — connect retrieval to the model (1.5 pt)

Model: `gpt-5.6-luna` for every call. Top 3 unless stated.

In [61]:
# Проверяем, какие переменные и функции из предыдущих частей доступны.

names_to_check = [
    "corpus",
    "fixed_queries",
    "bm25",
    "bm25_results",
    "embedding_model",
    "doc_embeddings",
    "query_embeddings",
    "client",
    "client_for",
    "chat",
    "tiktoken"
]

for name in names_to_check:
    print(f"{name}: {'available' if name in globals() else 'not found'}")

print("\nNumber of documents:", len(corpus) if "corpus" in globals() else "not found")
print("Number of queries:", len(fixed_queries) if "fixed_queries" in globals() else "not found")

corpus: available
fixed_queries: available
bm25: available
bm25_results: available
embedding_model: not found
doc_embeddings: not found
query_embeddings: available
client: available
client_for: not found
chat: not found
tiktoken: available

Number of documents: 74
Number of queries: 8


In [24]:
import os
from getpass import getpass
from openai import OpenAI

# Enter your existing OpenRouter API key securely
api_key = getpass("Enter your OpenRouter API key: ")

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=api_key
)

del api_key

print("OpenRouter client created successfully!")

Enter your OpenRouter API key:  ········


OpenRouter client created successfully!


In [62]:
# Отправляем тестовый запрос к модели через OpenRouter.
# Просим модель ответить определённой фразой, чтобы проверить подключение.
response = client.chat.completions.create(
    model="gpt-5.6-luna",
    messages=[
        {"role": "user", "content": "Reply with exactly: Connection successful"}
    ],
    max_tokens=20
)

print(response.choices[0].message.content)
print("Model:", response.model)

Connection successful
Model: openai/gpt-5.6-luna


## C1 — the complete prompt, printed before it is sent

In [79]:
import json
import numpy as np
import tiktoken
from sklearn.metrics.pairwise import cosine_similarity

# Загружаем вопросы из qa.json и выбираем первый вопрос C-01.
with open("data/qa.json", "r", encoding="utf-8") as f:
    qa = json.load(f)

selected_query = qa["questions"][0]
query_id = selected_query["id"]
question = selected_query["question"]

# Находим три наиболее подходящих документа с помощью EmbeddingGemma.
question_embedding = model.encode_query(
    [question],
    convert_to_numpy=True
)

scores = cosine_similarity(question_embedding, document_embeddings)[0]
top_indices = np.argsort(scores)[::-1][:3]

top_docs = [
    {"id": corpus[i]["id"], "score": float(scores[i])}
    for i in top_indices
]

docs_by_id = {doc["id"]: doc for doc in corpus}

# Создаём три пронумерованных контекстных блока.
context = "\n\n".join(
    f"[{rank}] Chunk ID: {item['id']}\n{docs_by_id[item['id']]['text']}"
    for rank, item in enumerate(top_docs, start=1)
)

# Создаём инструкции для модели согласно README.
system_prompt = """Answer the question using only the provided context.
If the answer cannot be found in the context, say that you do not know.
Cite the context blocks you used, like [1] or [2][3]."""

# Добавляем вопрос и найденные документы в запрос.
user_prompt = f"""Question: {question}

Retrieved context:
{context}"""

# Выводим полный prompt до отправки модели.
print("QUESTION ID:", query_id)

print("\nRETRIEVED DOCUMENTS:")
for rank, item in enumerate(top_docs, start=1):
    print(f"{rank}. {item['id']} | Score: {item['score']:.4f}")

print("\nSYSTEM PROMPT:")
print(system_prompt)

print("\nUSER PROMPT:")
print(user_prompt)

# Подсчитываем токены с помощью o200k_base.
encoding = tiktoken.get_encoding("o200k_base")

system_tokens = len(encoding.encode(system_prompt))
question_tokens = len(encoding.encode(question))
context_tokens = len(encoding.encode(context))
user_tokens = len(encoding.encode(user_prompt))

# Выводим количество токенов для каждой части запроса.
print("\nTOKEN COUNTS (o200k_base):")
print("System instructions:", system_tokens)
print("Question:", question_tokens)
print("Retrieved context:", context_tokens)
print("Complete user prompt:", user_tokens)
print("System + user prompt:", system_tokens + user_tokens)
print("Number of retrieved chunks:", len(top_docs))
print("Context characters:", len(context))

# Выводим количество токенов и символов каждого блока.
print("\nTOKENS PER CHUNK:")
for rank, item in enumerate(top_docs, start=1):
    doc_text = f"[{rank}] Chunk ID: {item['id']}\n{docs_by_id[item['id']]['text']}"
    print(
        item["id"],
        "| tokens:", len(encoding.encode(doc_text)),
        "| characters:", len(doc_text)
    )

QUESTION ID: C-01

RETRIEVED DOCUMENTS:
1. GO-04 | Score: 0.5845
2. GO-03 | Score: 0.4842
3. GO-15 | Score: 0.4506

SYSTEM PROMPT:
Answer the question using only the provided context.
If the answer cannot be found in the context, say that you do not know.
Cite the context blocks you used, like [1] or [2][3].

USER PROMPT:
Question: How much does a student in income band 2 receive, and when is it paid?

Retrieved context:
[1] Chunk ID: GO-04
The grant amount depends only on the income band, not on the GPA. A student in band 1 receives 250,000 tenge for the academic year. A student in band 2 receives 150,000 tenge for the academic year. The amount is paid in two equal instalments, the first in October and the second in February. There is no partial grant: a student either receives the full amount for their band or nothing.

[2] Chunk ID: GO-03
The income band is set from the monthly household income per person, declared on form GR-210. Band 1: below 85,000 tenge per person per month. Ban

In [80]:
# Отправляем вопрос и найденные документы модели через OpenRouter.
# Получаем ответ модели и информацию о количестве использованных токенов.
response = client.chat.completions.create(
    model="gpt-5.6-luna",
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ],
    max_tokens=300
)

# Выводим ответ, который сгенерировала модель
print("MODEL ANSWER:")
print(response.choices[0].message.content)

print("\nAPI TOKEN USAGE:")
print("Prompt tokens:", response.usage.prompt_tokens)
print("Completion tokens:", response.usage.completion_tokens)

# Считаем общее количество токенов запроса с помощью tiktoken
local_prompt_tokens = system_tokens + user_tokens

# Сравниваем локальный подсчёт токенов с результатом API
print("\nLOCAL TOKEN COUNT (o200k_base):")
print("System + user prompt:", local_prompt_tokens)
print("Difference:", response.usage.prompt_tokens - local_prompt_tokens)

MODEL ANSWER:
A student in income band 2 receives **150,000 tenge for the academic year**, paid in **two equal instalments of 75,000 tenge**: the first in **October** and the second in **February**. [1]

API TOKEN USAGE:
Prompt tokens: 386
Completion tokens: 56

LOCAL TOKEN COUNT (o200k_base):
System + user prompt: 376
Difference: 10


| Part of the input | Tokens (tiktoken `o200k_base`) |
|---|---:|
| System instructions | 44 |
| Question | 18 |
| Retrieved context | 309 |
| **Total input** | **376** |
| **`usage.prompt_tokens` from the response** | **386** |

- Number of retrieved chunks: 3
- Tokens per chunk: GO-04: 100, GO-03: 114, GO-15: 95
- Total context size: 309 tokens / 1193 characters

**Why do the tiktoken total and `usage.prompt_tokens` differ?**

> The local tiktoken count is 376 tokens, while the API reports 386 prompt tokens. The difference is 10 tokens. This happens because the API may count additional message-formatting and internal tokens that are not included in our local text count.

**What exactly does retrieval add to the model's input?**

> Retrieval adds the text of the three most relevant document chunks: GO-04, GO-03, and GO-15. These chunks contain 309 tokens and 1,193 characters in total. The model receives this context together with the question and system instructions, allowing it to answer using information from the corpus.

In [81]:
# Выводим количество токенов в инструкциях, вопросе и найденных документах
print("System instructions:", system_tokens)
print("Question:", question_tokens)
print("Retrieved context:", context_tokens)

# Показываем общее количество входных токенов по нашему подсчёту и по данным API
print("Total input (local):", system_tokens + user_tokens)
print("API prompt tokens:", response.usage.prompt_tokens)

# Проверяем количество найденных документов и длину контекста в символах
print("Number of retrieved chunks:", len(top_docs))
print("Context characters:", len(context))

System instructions: 44
Question: 18
Retrieved context: 309
Total input (local): 376
API prompt tokens: 386
Number of retrieved chunks: 3
Context characters: 1193


In [83]:
# Проверяем каждый из трёх найденных документов.
# Считаем количество токенов и символов в каждом блоке.
for rank, item in enumerate(top_docs, start=1):
    doc_text = f"[{rank}] Chunk ID: {item['id']}\n{docs_by_id[item['id']]['text']}"
    print(
        item["id"],
        "| tokens:", len(encoding.encode(doc_text)),
        "| characters:", len(doc_text)
    )

GO-04 | tokens: 100 | characters: 407
GO-03 | tokens: 114 | characters: 428
GO-15 | tokens: 95 | characters: 354


## C2 — three systems, six questions

In [68]:
import json

# Загружаем вопросы и правильные ответы из файла qa.json.
with open("data/qa.json", "r", encoding="utf-8") as f:
    qa = json.load(f)
    
# Выводим количество вопросов и всё содержимое файла
print("Количество вопросов:", len(qa["questions"]))
print(json.dumps(qa, ensure_ascii=False, indent=2))

Количество вопросов: 6
{
  "note": "Part C's six questions, each asked of all three systems. `reference` is what the corpus says; `relevant` lists the chunks that hold it. C-06 has no answer in the corpus on purpose: the right answer from a RAG system is that it does not know.",
  "questions": [
    {
      "id": "C-01",
      "question": "How much does a student in income band 2 receive, and when is it paid?",
      "reference": "150,000 tenge for the academic year, in two equal instalments: October and February.",
      "relevant": [
        "GO-04",
        "GO-15"
      ]
    },
    {
      "id": "C-02",
      "question": "My grant application was refused. How long do I have to appeal, and which form do I use?",
      "reference": "14 calendar days from the date on the decision letter, using form GR-305.",
      "relevant": [
        "GO-08",
        "GO-12",
        "GO-23"
      ]
    },
    {
      "id": "C-03",
      "question": "What is the minimum GPA needed to qualify for th

In [69]:
# Проверяем работу LLM без RAG.
# Отправляем каждый вопрос модели без найденных документов.
# Сохраняем ответы и сравниваем их с эталонными ответами из qa.json.

llm_only_results = {}

# Отправляем вопрос модели без дополнительного контекста
for item in qa["questions"]:
    response = client.chat.completions.create(
        model="openai/gpt-5.6-luna",
        messages=[
            {
                "role": "system",
                "content": "Answer the question briefly. If you do not know, say so."
            },
            {
                "role": "user",
                "content": item["question"]
            }
        ],
        max_tokens=200
    )

    # Извлекаем ответ модели и сохраняем его по ID вопроса.
    answer = response.choices[0].message.content
    llm_only_results[item["id"]] = answer

    # Выводим вопрос, ответ модели и эталонный ответ для сравнения.
    print(item["id"])
    print("Вопрос:", item["question"])
    print("Ответ модели:", answer)
    print("Эталонный ответ:", item["reference"])
    print("-" * 60)

C-01
Вопрос: How much does a student in income band 2 receive, and when is it paid?
Ответ модели: Which student funding scheme or institution do you mean? “Income band 2” varies by country and program, so I need that context to give the amount and payment date.
Эталонный ответ: 150,000 tenge for the academic year, in two equal instalments: October and February.
------------------------------------------------------------
C-02
Вопрос: My grant application was refused. How long do I have to appeal, and which form do I use?
Ответ модели: Appeal deadlines and forms vary by grant program and jurisdiction. Check your refusal letter—it should state the deadline and the required form or appeal process. If you share the grant name, agency, and country, I can help identify the exact requirements.
Эталонный ответ: 14 calendar days from the date on the decision letter, using form GR-305.
------------------------------------------------------------
C-03
Вопрос: What is the minimum GPA needed to qua

In [70]:
# Проверяем BM25 RAG.
# Для каждого вопроса находим 3 наиболее релевантных документа.
# Передаём найденный контекст модели и сравниваем ответ с эталоном.

bm25_rag_results = {}

# Создаём словарь документов для быстрого поиска по ID.
docs_by_id = {doc["id"]: doc for doc in corpus}

for item in qa["questions"]:
    # Токенизируем вопрос и вычисляем BM25-оценки документов
    query_tokens = tokenize(item["question"])
    scores = bm25.get_scores(query_tokens)

    # Выбираем три документа с наибольшими оценками
    top_indices = sorted(
        range(len(scores)),
        key=lambda i: scores[i],
        reverse=True
    )[:3]

    top_docs = [corpus[i] for i in top_indices]

    # Формируем контекст из найденных документов
    context = "\n\n".join(
        f"[{doc['id']}]\n{doc['text']}"
        for doc in top_docs
    )

    # Отправляем вопрос и контекст модели.
    response = client.chat.completions.create(
        model="openai/gpt-5.6-luna",
        messages=[
            {
                "role": "system",
                "content": (
                    "Answer using only the provided context. "
                    "If the context does not contain the answer, "
                    "say that you do not know. Do not invent facts."
                )
            },
            {
                "role": "user",
                "content": (
                    f"Question: {item['question']}\n\n"
                    f"Retrieved context:\n{context}"
                )
            }
        ],
        max_tokens=200
    )

    # Сохраняем найденные документы и ответ модели
    answer = response.choices[0].message.content

    bm25_rag_results[item["id"]] = {
        "answer": answer,
        "retrieved_ids": [doc["id"] for doc in top_docs],
        "reference": item["reference"],
        "relevant": item["relevant"]
    }

    print(item["id"])
    print("Вопрос:", item["question"])
    print("Найденные документы:", [doc["id"] for doc in top_docs])
    print("Ответ BM25 RAG:", answer)
    print("Эталонный ответ:", item["reference"])
    print("-" * 60)

C-01
Вопрос: How much does a student in income band 2 receive, and when is it paid?
Найденные документы: ['GO-04', 'GO-02', 'GO-01']
Ответ BM25 RAG: A student in income band 2 receives **150,000 tenge for the academic year**, paid in **two equal instalments of 75,000 tenge**: the first in **October** and the second in **February**.
Эталонный ответ: 150,000 tenge for the academic year, in two equal instalments: October and February.
------------------------------------------------------------
C-02
Вопрос: My grant application was refused. How long do I have to appeal, and which form do I use?
Найденные документы: ['GO-05', 'GO-08', 'GO-12']
Ответ BM25 RAG: You have **14 calendar days from the date on the decision letter** to appeal. Use **form GR-305**.
Эталонный ответ: 14 calendar days from the date on the decision letter, using form GR-305.
------------------------------------------------------------
C-03
Вопрос: What is the minimum GPA needed to qualify for the need-based grant?
Найд

In [71]:
# Проверяем Embedding RAG.
# Для каждого вопроса сравниваем его embedding с embedding всех документов.
# Передаём модели три наиболее похожих документа и сохраняем ответ.

embedding_rag_results = {}

for item in qa["questions"]:
    # Преобразуем вопрос в вектор с помощью EmbeddingGemma
    question_embedding = model.encode_query(
        [item["question"]],
        convert_to_numpy=True
    )

    # Вычисляем cosine similarity между вопросом и документами
    scores = cosine_similarity(
        question_embedding,
        document_embeddings
    )[0]

    # Выбираем три документа с наибольшим сходством
    top_indices = np.argsort(scores)[::-1][:3]
    top_docs = [corpus[i] for i in top_indices]

    # Формируем контекст из найденных документов
    context = "\n\n".join(
        f"[{doc['id']}]\n{doc['text']}"
        for doc in top_docs
    )

    # Отправляем вопрос и контекст модели
    response = client.chat.completions.create(
        model="openai/gpt-5.6-luna",
        messages=[
            {
                "role": "system",
                "content": (
                    "Answer using only the provided context. "
                    "If the context does not contain the answer, "
                    "say that you do not know. Do not invent facts."
                )
            },
            {
                "role": "user",
                "content": (
                    f"Question: {item['question']}\n\n"
                    f"Retrieved context:\n{context}"
                )
            }
        ],
        max_tokens=200
    )

    # Сохраняем ответ, найденные документы и эталонный ответ
    answer = response.choices[0].message.content

    embedding_rag_results[item["id"]] = {
        "answer": answer,
        "retrieved_ids": [doc["id"] for doc in top_docs],
        "reference": item["reference"],
        "relevant": item["relevant"]
    }

    print(item["id"])
    print("Вопрос:", item["question"])
    print("Найденные документы:", [doc["id"] for doc in top_docs])
    print("Ответ Embedding RAG:", answer)
    print("Эталонный ответ:", item["reference"])
    print("-" * 60)

C-01
Вопрос: How much does a student in income band 2 receive, and when is it paid?
Найденные документы: ['GO-04', 'GO-03', 'GO-15']
Ответ Embedding RAG: A student in income band 2 receives **150,000 tenge for the academic year**, paid in **two equal instalments of 75,000 tenge**: one in **October** and one in **February**.
Эталонный ответ: 150,000 tenge for the academic year, in two equal instalments: October and February.
------------------------------------------------------------
C-02
Вопрос: My grant application was refused. How long do I have to appeal, and which form do I use?
Найденные документы: ['GO-08', 'GO-12', 'GO-11']
Ответ Embedding RAG: You have **14 calendar days from the date on the decision letter** to appeal. Use **form GR-305**.
Эталонный ответ: 14 calendar days from the date on the decision letter, using form GR-305.
------------------------------------------------------------
C-03
Вопрос: What is the minimum GPA needed to qualify for the need-based grant?
Найденн

In [84]:
# Собираем ответы трёх систем и эталонные ответы.
# Формируем таблицу для сравнения результатов.
# Правильность ответов оцениваем отдельно.

comparison_results = []

for item in qa["questions"]:
    qid = item["id"]
    reference = item["reference"]
    
    # Получаем ответы трёх систем для каждого вопроса
    llm_answer = llm_only_results[qid]
    bm25_answer = bm25_rag_results[qid]["answer"]
    embedding_answer = embedding_rag_results[qid]["answer"]

    comparison_results.append({
        "Question": qid,
        "Reference": reference,
        "LLM only": llm_answer,
        "BM25 RAG": bm25_answer,
        "Embedding RAG": embedding_answer
    })

# Создаём и выводим итоговую таблицу
import pandas as pd

comparison_df = pd.DataFrame(comparison_results)
display(comparison_df)

,Question,Reference,LLM only,BM25 RAG,Embedding RAG
0,C-01,"150,000 tenge for the academic year, in two eq...",Which student funding scheme or institution do...,"A student in income band 2 receives **150,000 ...","A student in income band 2 receives **150,000 ..."
1,C-02,14 calendar days from the date on the decision...,Appeal deadlines and forms vary by grant progr...,You have **14 calendar days from the date on t...,You have **14 calendar days from the date on t...
2,C-03,A cumulative GPA of at least 2.67 on the 4.0 s...,It depends on the specific need-based grant an...,The minimum GPA needed is **2.67 on a 4.0 scal...,The minimum GPA is **2.67 on a 4.0 scale** for...
3,C-04,A presidential decree of October 2017 ordered ...,NaN,Kazakhstan decided to switch from Cyrillic to ...,Kazakhstan decided to switch from Cyrillic to ...
4,C-05,"In 1963, as the Almaty Institute of National E...",Narxoz University was founded in **1963** as t...,"Narxoz University was founded in 1963, origina...",Narxoz University was founded in 1963 as the *...
5,C-06,The corpus does not say. GO-20 says master's s...,"It depends on your country, school, program, i...",I don’t know the amount for master’s students....,The provided information does not specify the ...


Mark each answer ✓ (agrees with `reference`) or ✗, and give a few words of the answer.

| Question | LLM only | BM25 RAG | Embedding RAG |
|---|---|---|---|
| C-01 | ✗ — Asked for more context. | ✓ — 150,000 tenge; October and February. | ✓ — 150,000 tenge; October and February. |
| C-02 | ✗ — Did not provide the deadline or form. | ✓ — 14 days; form GR-305. | ✓ — 14 days; form GR-305. |
| C-03 | ✗ — Did not provide the minimum GPA. | ✓ — Minimum GPA 2.67. | ✓ — Minimum GPA 2.67. |
| C-04 | ✗ — Answer was incomplete. | ✓ — October 2017; deadline moved to 2031. | ✓ — October 2017; deadline moved to 2031. |
| C-05 | ✓ — Founded in 1963 as the Alma-Ata Institute of National Economy. | ✓ — Founded in 1963 as the Almaty Institute of National Economy. | ✓ — Founded in 1963 as the Almaty Institute of National Economy. |
| C-06 | ✗ — Gave unsupported information about US grants. | ✓ — Master's students have a separate scheme; amount not specified. | ✓ — Master's grant amount not specified. |
| **Agrees with reference** | **1/6** | **6/6** | **6/6** |

## C3 — question 1 of 3: (C-01)

**1. What BM25 retrieved (top 3, ids and scores):**

> GO-04 (20.2192), GO-02 (18.7422), GO-01 (17.2430).

**2. What EmbeddingGemma retrieved (top 3, ids and scores):**

> GO-04 (0.5845), GO-03 (0.4842), GO-15 (0.4506).

**3. What was sent to the model (the context blocks):**

> **BM25 RAG — GO-04, GO-02, GO-01**

> **[GO-04]**
>
> The grant amount depends only on the income band, not on the GPA. A student in band 1 receives 250,000 tenge for the academic year. A student in band 2 receives 150,000 tenge for the academic year. The amount is paid in two equal instalments, the first in October and the second in February. There is no partial grant: a student either receives the full amount for their band or nothing.

> **[GO-02]**
>
> An applicant qualifies for the 2026 grant when all three conditions hold at the time of the decision: their cumulative GPA is at least 2.67 on the 4.0 scale; their household is in income band 1 or income band 2; and both a transcript and a copy of the national id card are on file with the Grant Office. Meeting two conditions out of three is not enough. An applicant who is not on the Grant Office record does not qualify, whatever their message says.

> **[GO-01]**
>
> The need-based study grant 2026 is run by the Grant Office for full-time undergraduate students. It exists so that a student from a low-income household can pay for study costs without taking on paid work that harms their studies. One grant is awarded per student per academic year. The grant is a gift, not a loan: it does not have to be paid back unless the student withdraws early, as explained in the section on academic progress.

> **Embedding RAG — GO-04, GO-03, GO-15**

> **[GO-04]**
>
> The grant amount depends only on the income band, not on the GPA. A student in band 1 receives 250,000 tenge for the academic year. A student in band 2 receives 150,000 tenge for the academic year. The amount is paid in two equal instalments, the first in October and the second in February. There is no partial grant: a student either receives the full amount for their band or nothing.

> **[GO-03]**
>
> The income band is set from the monthly household income per person, declared on form GR-210. Band 1: below 85,000 tenge per person per month. Band 2: from 85,000 up to 150,000 tenge per person per month. Band 3: above 150,000 tenge per person per month. Households in band 3 are not eligible for the need-based grant, although their students may still apply for merit scholarships, which are run separately.

> **[GO-15]**
>
> Грант выплачивается двумя равными частями — в октябре и в феврале — на банковский счёт, указанный студентом в форме GR-104. Счёт должен быть открыт на имя самого студента: перевод на счёт родителей или других лиц невозможен. Если реквизиты изменились, студент обязан обновить их в портале до 1 октября или до 1 февраля соответственно.

**4. How retrieval changed the answer:**

> Both BM25 RAG and EmbeddingGemma RAG retrieved GO-04, which contains the exact grant amount and payment schedule. Both systems correctly answered that a student in income band 2 receives 150,000 tenge per academic year, paid in two instalments of 75,000 tenge in October and February. EmbeddingGemma also retrieved GO-15, which confirms the payment months. Retrieval provided specific evidence that the model could use instead of relying only on its general knowledge.

**5. What was missed, and was it a retrieval problem or a generation problem? Quote the evidence.**

> Neither RAG system missed the main information. GO-04 states: "A student in band 2 receives 150,000 tenge for the academic year." It also states that the amount is paid in two equal instalments in October and February. Both systems retrieved this document, so there was no significant retrieval problem. Both answers included the required information, so there was no significant generation problem either.

In [85]:
# Для C-01 выводим топ-3 документа BM25 и EmbeddingGemma с оценками.
# Показываем полный контекст, переданный модели обеими RAG-системами.

q = next(item for item in qa["questions"] if item["id"] == "C-01")
question = q["question"]

print("QUESTION:", q["id"], question)

# 1.BM25: рассчитываем оценки и выбираем три лучших документа
bm25_scores = bm25.get_scores(tokenize(question))
bm25_top = sorted(
    range(len(bm25_scores)),
    key=lambda i: bm25_scores[i],
    reverse=True
)[:3]

print("\n1. BM25 TOP 3:")
for i in bm25_top:
    print(f"{corpus[i]['id']} | Score: {bm25_scores[i]:.4f}")

# 2.EmbeddingGemma: вычисляем cosine similarity
q_embedding = model.encode_query(
    [question],
    convert_to_numpy=True
)
embedding_scores = cosine_similarity(
    q_embedding, document_embeddings
)[0]
embedding_top = np.argsort(embedding_scores)[::-1][:3]

print("\n2. EMBEDDINGGEMMA TOP 3:")
for i in embedding_top:
    print(f"{corpus[i]['id']} | Score: {embedding_scores[i]:.4f}")

# 3.Выводим контекст, который реально использовался в ранее выполненных запросах к модели.
for system_name, results in [
    ("BM25 RAG", bm25_rag_results),
    ("Embedding RAG", embedding_rag_results)
]:
    print(f"\n3. CONTEXT SENT TO {system_name}:")
    for doc_id in results["C-01"]["retrieved_ids"]:
        doc = next(d for d in corpus if d["id"] == doc_id)
        print(f"\n[{doc_id}]\n{doc['text']}")

# Показываем ответы обеих систем для анализа пунктов 4 и 5
print("\n4–5. ANSWERS:")
print("BM25 RAG:", bm25_rag_results["C-01"]["answer"])
print("Embedding RAG:", embedding_rag_results["C-01"]["answer"])
print("Reference:", q["reference"])

QUESTION: C-01 How much does a student in income band 2 receive, and when is it paid?

1. BM25 TOP 3:
GO-04 | Score: 20.2192
GO-02 | Score: 18.7422
GO-01 | Score: 17.2430

2. EMBEDDINGGEMMA TOP 3:
GO-04 | Score: 0.5845
GO-03 | Score: 0.4842
GO-15 | Score: 0.4506

3. CONTEXT SENT TO BM25 RAG:

[GO-04]
The grant amount depends only on the income band, not on the GPA. A student in band 1 receives 250,000 tenge for the academic year. A student in band 2 receives 150,000 tenge for the academic year. The amount is paid in two equal instalments, the first in October and the second in February. There is no partial grant: a student either receives the full amount for their band or nothing.

[GO-02]
An applicant qualifies for the 2026 grant when all three conditions hold at the time of the decision: their cumulative GPA is at least 2.67 on the 4.0 scale; their household is in income band 1 or income band 2; and both a transcript and a copy of the national id card are on file with the Grant Offic

## C3 — question 2 of 3: C-02

**1. What BM25 retrieved (top 3, ids and scores):**

> GO-05 (17.2414), GO-08 (14.5287), GO-12 (13.5735).

**2. What EmbeddingGemma retrieved (top 3, ids and scores):**

> GO-08 (0.6029), GO-12 (0.5591), GO-11 (0.5343).

**3. What was sent to the model (the context blocks):**

> **BM25 RAG — GO-05, GO-08, GO-12**

> **[GO-05]**
>
> ARCHIVED — kept for reference only. Under the 2025 scheme the minimum GPA for the grant was 3.0, band 1 received 200,000 tenge, band 2 received 120,000 tenge, and the whole amount was paid in a single instalment in November. These rules were replaced on 1 June 2026 and do not apply to any application made for the 2026 academic year.

> **[GO-08]**
>
> Form GR-305 is used to appeal a decision of the Grant Office. It is the appeal form and nothing else: it cannot be used to apply for the first time or to report a change in circumstances. On GR-305 the student states which part of the decision they believe is wrong and which rule in this handbook supports their case.

> **[GO-12]**
>
> A student who disagrees with a decision may appeal within 14 calendar days of the date on the decision letter, using form GR-305. The appeal is read by a panel of three staff members who did not take part in the original decision, and the panel answers within 20 working days. An appeal reviews the decision on the evidence that was on file; new income information cannot be added through an appeal and needs a new application.

> **Embedding RAG — GO-08, GO-12, GO-11**

> **[GO-08]**
>
> Form GR-305 is used to appeal a decision of the Grant Office. It is the appeal form and nothing else: it cannot be used to apply for the first time or to report a change in circumstances. On GR-305 the student states which part of the decision they believe is wrong and which rule in this handbook supports their case.

> **[GO-12]**
>
> A student who disagrees with a decision may appeal within 14 calendar days of the date on the decision letter, using form GR-305. The appeal is read by a panel of three staff members who did not take part in the original decision, and the panel answers within 20 working days. An appeal reviews the decision on the evidence that was on file; new income information cannot be added through an appeal and needs a new application.

> **[GO-11]**
>
> If a required document is missing, the Grant Office does not refuse the application straight away. The decision is recorded as 'more information needed', and the student is told exactly which document is missing. The student then has 10 working days to upload it through the portal. If the document does not arrive in that time, the application lapses and the student must apply again before the deadline.

**4. How retrieval changed the answer:**

> Both RAG systems gave accurate and specific answers. They identified the appeal deadline as 14 calendar days from the date on the decision letter and the correct form, GR-305. Embedding RAG retrieved the two most directly relevant documents first, while BM25 also retrieved GO-05, which contains archived 2025 rules. The archived document did not affect the final answer.

**5. What was missed, and was it a retrieval problem or a generation problem? Quote the evidence.**

> Both RAG systems gave the correct answer. A student can appeal within 14 calendar days using form GR-305. However, BM25 also retrieved GO-05, which contains old grant rules from 2025. This was a small retrieval problem because the document was not related to the question. The model used the correct information from GO-08 and GO-12, so there was no generation problem. Evidence: GO-12 states that students can appeal within 14 calendar days using form GR-305.

In [75]:
# Для C-02 выводим топ-3 документа BM25 и EmbeddingGemma с оценками.
# Показываем полный контекст, ответы обеих RAG-систем и эталонный ответ.

q = next(item for item in qa["questions"] if item["id"] == "C-02")
question = q["question"]

print("QUESTION:", q["id"], question)

# 1.BM25: рассчитываем оценки документов и выбираем top-3
bm25_scores = bm25.get_scores(tokenize(question))
bm25_top = sorted(
    range(len(bm25_scores)),
    key=lambda i: bm25_scores[i],
    reverse=True
)[:3]

print("\n1. BM25 TOP 3:")
for i in bm25_top:
    print(f"{corpus[i]['id']} | Score: {bm25_scores[i]:.4f}")

# 2.EmbeddingGemma: вычисляем cosine similarity и выбираем top-3
q_embedding = model.encode_query(
    [question],
    convert_to_numpy=True
)
embedding_scores = cosine_similarity(
    q_embedding, document_embeddings
)[0]
embedding_top = np.argsort(embedding_scores)[::-1][:3]

print("\n2. EMBEDDINGGEMMA TOP 3:")
for i in embedding_top:
    print(f"{corpus[i]['id']} | Score: {embedding_scores[i]:.4f}")

# 3.Выводим полные тексты документов, которые фактически использовались в предыдущих RAG-запросах.
for system_name, results in [
    ("BM25 RAG", bm25_rag_results),
    ("Embedding RAG", embedding_rag_results)
]:
    print(f"\n3. CONTEXT SENT TO {system_name}:")
    for doc_id in results["C-02"]["retrieved_ids"]:
        doc = next(d for d in corpus if d["id"] == doc_id)
        print(f"\n[{doc_id}]\n{doc['text']}")

# 4–5. Выводим ответы моделей и эталон для сравнения
print("\n4–5. ANSWERS:")
print("\nBM25 RAG:", bm25_rag_results["C-02"]["answer"])
print("\nEmbedding RAG:", embedding_rag_results["C-02"]["answer"])
print("\nReference:", q["reference"])

QUESTION: C-02 My grant application was refused. How long do I have to appeal, and which form do I use?

1. BM25 TOP 3:
GO-05 | Score: 17.2414
GO-08 | Score: 14.5287
GO-12 | Score: 13.5735

2. EMBEDDINGGEMMA TOP 3:
GO-08 | Score: 0.6029
GO-12 | Score: 0.5591
GO-11 | Score: 0.5343

3. CONTEXT SENT TO BM25 RAG:

[GO-05]
ARCHIVED — kept for reference only. Under the 2025 scheme the minimum GPA for the grant was 3.0, band 1 received 200,000 tenge, band 2 received 120,000 tenge, and the whole amount was paid in a single instalment in November. These rules were replaced on 1 June 2026 and do not apply to any application made for the 2026 academic year.

[GO-08]
Form GR-305 is used to appeal a decision of the Grant Office. It is the appeal form and nothing else: it cannot be used to apply for the first time or to report a change in circumstances. On GR-305 the student states which part of the decision they believe is wrong and which rule in this handbook supports their case.

[GO-12]
A studen

## C3 — question 3 of 3: C-03

**1. What BM25 retrieved (top 3, ids and scores):**

> GO-16 (15.7985), GO-02 (14.0587), GO-06 (13.8516).

**2. What EmbeddingGemma retrieved (top 3, ids and scores):**

> GO-02 (0.6286), GO-20 (0.5691), GO-05 (0.5451).

**3. What was sent to the model (the context blocks):**

> **BM25 RAG — GO-16, GO-02, GO-06**

> **[GO-16]**
>
> The need-based grant can be held together with a merit scholarship, but not together with a second need-based grant from any other source. The total of all university support a student receives in one academic year is capped at 400,000 tenge. When the cap would be exceeded, the need-based grant is reduced so that the total equals the cap.

> **[GO-02]**
>
> An applicant qualifies for the 2026 grant when all three conditions hold at the time of the decision: their cumulative GPA is at least 2.67 on the 4.0 scale; their household is in income band 1 or income band 2; and both a transcript and a copy of the national id card are on file with the Grant Office. Meeting two conditions out of three is not enough. An applicant who is not on the Grant Office record does not qualify, whatever their message says.

> **[GO-06]**
>
> Form GR-104 is the application form for the need-based study grant. It is the only way to apply; an email or a letter is not an application. GR-104 is filled in on the student portal and asks for the student's id, programme, the bank account for payment, and a declaration that the information is true. The transcript and the id card copy are uploaded as attachments to GR-104.

> **Embedding RAG — GO-02, GO-20, GO-05**

> **[GO-02]**
>
> An applicant qualifies for the 2026 grant when all three conditions hold at the time of the decision: their cumulative GPA is at least 2.67 on the 4.0 scale; their household is in income band 1 or income band 2; and both a transcript and a copy of the national id card are on file with the Grant Office. Meeting two conditions out of three is not enough. An applicant who is not on the Grant Office record does not qualify, whatever their message says.

> **[GO-20]**
>
> The need-based grant is for full-time undergraduate students only. Part-time students, distance-learning students and incoming exchange students cannot apply. Master's and doctoral students are supported by a separate scheme, which this handbook does not describe; they should ask their own department for its rules.

> **[GO-05]**
>
> ARCHIVED — kept for reference only. Under the 2025 scheme the minimum GPA for the grant was 3.0, band 1 received 200,000 tenge, band 2 received 120,000 tenge, and the whole amount was paid in a single instalment in November. These rules were replaced on 1 June 2026 and do not apply to any application made for the 2026 academic year.

**4. How retrieval changed the answer:**

> Both RAG systems provided accurate and specific answers. They identified the minimum cumulative GPA as 2.67 on a 4.0 scale. BM25 retrieved GO-02 in second place, while EmbeddingGemma retrieved it first. EmbeddingGemma also retrieved GO-05, which contains the archived 2025 GPA requirement of 3.0, but the model correctly used the current 2026 rule from GO-02.

**5. What was missed, and was it a retrieval problem or a generation problem? Quote the evidence.**

> Both RAG systems found GO-02 and gave the correct answer: the minimum GPA is 2.67. However, EmbeddingGemma also retrieved GO-05, which contains the old GPA requirement of 3.0 from 2025. This was a small retrieval problem because the document was outdated. The model used the correct information from GO-02, so there was no generation problem. Evidence: GO-02 states that the minimum GPA is "at least 2.67 on the 4.0 scale."

In [77]:
# Для C-03 выводим топ-3 документа BM25 и EmbeddingGemma с оценками.
# Показываем полный контекст, ответы обеих RAG-систем и эталонный ответ.

q = next(item for item in qa["questions"] if item["id"] == "C-03")
question = q["question"]

print("QUESTION:", q["id"], question)

# 1.BM25: рассчитываем оценки документов и выбираем top-3
bm25_scores = bm25.get_scores(tokenize(question))
bm25_top = sorted(
    range(len(bm25_scores)),
    key=lambda i: bm25_scores[i],
    reverse=True
)[:3]

print("\n1. BM25 TOP 3:")
for i in bm25_top:
    print(f"{corpus[i]['id']} | Score: {bm25_scores[i]:.4f}")

# 2.EmbeddingGemma: вычисляем cosine similarity и выбираем top-3
q_embedding = model.encode_query(
    [question],
    convert_to_numpy=True
)
embedding_scores = cosine_similarity(
    q_embedding, document_embeddings
)[0]
embedding_top = np.argsort(embedding_scores)[::-1][:3]

print("\n2. EMBEDDINGGEMMA TOP 3:")
for i in embedding_top:
    print(f"{corpus[i]['id']} | Score: {embedding_scores[i]:.4f}")

# 3.Выводим полные тексты документов, которые использовались в ранее выполненных RAG-запросах
for system_name, results in [
    ("BM25 RAG", bm25_rag_results),
    ("Embedding RAG", embedding_rag_results)
]:
    print(f"\n3. CONTEXT SENT TO {system_name}:")
    for doc_id in results["C-03"]["retrieved_ids"]:
        doc = next(d for d in corpus if d["id"] == doc_id)
        print(f"\n[{doc_id}]\n{doc['text']}")

print("\n4–5. ANSWERS:")
print("\nBM25 RAG:", bm25_rag_results["C-03"]["answer"])
print("\nEmbedding RAG:", embedding_rag_results["C-03"]["answer"])
print("\nReference:", q["reference"])

QUESTION: C-03 What is the minimum GPA needed to qualify for the need-based grant?

1. BM25 TOP 3:
GO-16 | Score: 15.7985
GO-02 | Score: 14.0587
GO-06 | Score: 13.8516

2. EMBEDDINGGEMMA TOP 3:
GO-02 | Score: 0.6286
GO-20 | Score: 0.5691
GO-05 | Score: 0.5451

3. CONTEXT SENT TO BM25 RAG:

[GO-16]
The need-based grant can be held together with a merit scholarship, but not together with a second need-based grant from any other source. The total of all university support a student receives in one academic year is capped at 400,000 tenge. When the cap would be exceeded, the need-based grant is reduced so that the total equals the cap.

[GO-02]
An applicant qualifies for the 2026 grant when all three conditions hold at the time of the decision: their cumulative GPA is at least 2.67 on the 4.0 scale; their household is in income band 1 or income band 2; and both a transcript and a copy of the national id card are on file with the Grant Office. Meeting two conditions out of three is not enou

---
# Final questions

**1. What is the main difference between BM25 and embedding retrieval?**

> BM25 finds documents based mainly on exact word matches and term frequency. Embedding retrieval compares the semantic meaning of the query and documents using vectors.

**2. When can BM25 perform better than embeddings?**

> BM25 can perform better when the query contains exact terms, form numbers, IDs, or specific keywords, such as GR-305.

**3. When can embeddings perform better than BM25?**

> Embeddings can perform better when the question uses different words but has a similar meaning to the relevant document.

**4. Why is cosine similarity used with embeddings?**

> Cosine similarity measures how similar two vectors are by comparing the angle between them. It helps retrieve documents that are semantically similar to the query.

**5. What information is actually sent to an LLM in a RAG system?**

> The LLM receives the system instructions, the user's question, and the text of the retrieved document chunks.

**6. Does the LLM search the original documents itself?**

> No. The retrieval system searches the document collection and selects relevant chunks. The LLM generates an answer using the question and the retrieved context.

**7. What happens if retrieval returns an irrelevant chunk?**

> The LLM may receive misleading or unnecessary information and produce an incorrect or less relevant answer. The result depends on the retrieved context and how the model uses it.

**8. Why can increasing Top-K sometimes improve RAG and sometimes make it worse? Report K = 1, 3 and 8 on one question.**

> I tested Embedding RAG on question C-03 about the minimum GPA.
>
> - **K = 1:** Retrieved GO-02. The answer correctly stated that the minimum GPA is 2.67 on a 4.0 scale.
> - **K = 3:** Retrieved GO-02, GO-20, and GO-05. The answer was still correct and specified that the rule applies to the 2026 need-based grant.
> - **K = 8:** Retrieved eight documents, including GO-05, which contains the archived 2025 GPA requirement of 3.0. The model still gave the correct answer: 2.67.
>
> Increasing Top-K can improve RAG by providing more relevant information and context. However, it can also introduce irrelevant or outdated information, making the answer less accurate or more confusing. In this experiment, all three settings produced the correct answer, but K = 1 provided sufficient information with less unnecessary context.

In [86]:
# Question 8: проверяем влияние Top-K на качество RAG.
# Используем вопрос C-03 о минимальном GPA.
# Сравниваем количество найденных документов: K = 1, 3 и 8.

q = next(item for item in qa["questions"] if item["id"] == "C-03")
question = q["question"]

# Преобразуем вопрос в embedding и рассчитываем сходство с документами.
q_embedding = model.encode_query(
    [question],
    convert_to_numpy=True
)

scores = cosine_similarity(
    q_embedding, document_embeddings
)[0]

# Сортируем документы от наиболее похожих к наименее похожим.
ranked_indices = np.argsort(scores)[::-1]

# Проверяем три значения K.
for k in [1, 3, 8]:
    top_indices = ranked_indices[:k]

    context = "\n\n".join(
        f"[{corpus[i]['id']}]\n{corpus[i]['text']}"
        for i in top_indices
    )

    print(f"\n{'=' * 50}")
    print(f"K = {k}")
    print("Retrieved documents:")

    for i in top_indices:
        print(f"{corpus[i]['id']} | Score: {scores[i]:.4f}")

    # Отправляем вопрос и найденный контекст модели
    response = client.chat.completions.create(
        model="openai/gpt-5.6-luna",
        messages=[
            {
                "role": "system",
                "content": (
                    "Answer using only the provided context. "
                    "If the context does not contain enough information, "
                    "say that you do not know. Do not invent facts."
                )
            },
            {
                "role": "user",
                "content": (
                    f"Question: {question}\n\n"
                    f"Retrieved context:\n{context}"
                )
            }
        ],
        max_tokens=200
    )

    print("\nModel answer:")
    print(response.choices[0].message.content)
    print("\nReference:")
    print(q["reference"])


K = 1
Retrieved documents:
GO-02 | Score: 0.6286

Model answer:
The minimum GPA is **2.67 on a 4.0 scale**.

Reference:
A cumulative GPA of at least 2.67 on the 4.0 scale. (The 3.0 in GO-05 is the archived 2025 rule.)

K = 3
Retrieved documents:
GO-02 | Score: 0.6286
GO-20 | Score: 0.5691
GO-05 | Score: 0.5451

Model answer:
The minimum GPA is **2.67 on a 4.0 scale** for the 2026 need-based grant.

Reference:
A cumulative GPA of at least 2.67 on the 4.0 scale. (The 3.0 in GO-05 is the archived 2025 rule.)

K = 8
Retrieved documents:
GO-02 | Score: 0.6286
GO-20 | Score: 0.5691
GO-05 | Score: 0.5451
GO-16 | Score: 0.5378
GO-01 | Score: 0.5347
GO-06 | Score: 0.5330
GO-19 | Score: 0.5265
GO-17 | Score: 0.5252

Model answer:
The minimum GPA needed is **2.67 on a 4.0 scale**.

Reference:
A cumulative GPA of at least 2.67 on the 4.0 scale. (The 3.0 in GO-05 is the archived 2025 rule.)


**9. Is a vector database required to build a RAG system?**

> No. A vector database is not strictly required. For a small corpus, embeddings can be stored in a NumPy array and compared using cosine similarity. A vector database becomes useful when the collection is large and requires efficient storage and retrieval.

**10. Which retrieval approach worked better for this corpus, and what evidence supports that?** Give hit@3 for both over your ten queries and the C2 agreement counts for both RAG systems.

> Embedding retrieval performed better on the ten-query retrieval evaluation. BM25 achieved hit@3 of 7/10 (70%), while EmbeddingGemma achieved 10/10 (100%). In Part C2, both BM25 RAG and Embedding RAG agreed with all six reference answers, achieving 6/6 each. Therefore, EmbeddingGemma retrieved relevant documents more consistently in the ten-query test, although both RAG systems produced correct answers for all six C2 questions.

---
# Conclusion

> This assignment compared BM25 and EmbeddingGemma in a RAG system. EmbeddingGemma performed better than BM25, with 100% hit@3 compared to 70%. Both RAG systems answered all six questions correctly, while the LLM without RAG answered only one correctly. The Top-K experiment showed that K = 1, 3, and 8 gave the correct answer. However, more documents can include old or unnecessary information. Overall, RAG helped the model give better answers using information from the documents. Next, f filter can be added to remove outdated documents because old information can lead to incorrect answers.